In [ ]:
import numpy as np
import random
import heapq
import time
import tracemalloc

from collections import deque
import pandas as pd

**CREATE MAZE**

In [ ]:
def create_maze(dim, exits, i):
    random.seed(i)      # Set a fixed random seed to ensure reproducibility
                        # The same seed and input parameters generate the same maze.
    # Create a grid filled with walls
    maze = np.full((dim * 2 + 1, dim * 2 + 1), "#", dtype=str)
    # Start cell
    x, y = (0, 0)
    maze[2 * x + 1, 2 * y + 1] = "."
    # DFS Backtracking
    stack = [(x, y)]
    while stack:
        x, y = stack[-1]
        directions = [(0, 1),(1, 0),(0, -1),(-1, 0)]
        random.shuffle(directions)   # Randomly shuffle movement directions to generate diverse maze structures.
        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if ( 0 <= nx < dim
                and 0 <= ny < dim
                and maze[2 * nx + 1, 2 * ny + 1] == "#"):
                maze[2 * nx + 1, 2 * ny + 1] = "."
                maze[2 * x + 1 + dx,
                    2 * y + 1 + dy] = "."
                stack.append((nx, ny))
                break
        else:
            stack.pop()

    # Add random openings
    n = len(maze)
    for _ in range(dim // 2):
        r = random.randint(1, n - 2)   # Randomly select an internal wall position to potentially create an additional passage.
        c = random.randint(1, n - 2)
        if maze[r, c] == "#":
            for dr, dc in [(1, 0),(-1, 0),(0, 1),(0, -1)]:
                if maze[r + dr, c + dc] == ".":
                    maze[r, c] = "."
                    break
    # Add entrance
    maze[1, 0] = "S"
    # Find possible exit positions
    border_cells = []
    for r in range(n):
        for c in range(n):
            if ((r in [0, n - 1] or c in [0, n - 1])
                and maze[r, c] == "#"):
                for dr, dc in [(1, 0),(-1, 0),(0, 1),(0, -1)]:
                    nr, nc = r + dr, c + dc
                    if (0 <= nr < n and 0 <= nc < n and maze[nr, nc] == "."):
                        border_cells.append((r, c))
                        break
    # Randomly select exits
    chosen = random.sample(border_cells, exits)    # Randomly select distinct boundary cells as maze exits.
    for r, c in chosen:
        maze[r, c] = "E"
    return maze

In [ ]:
maze = create_maze(10, 2, 42)

for row in maze:
    print("".join(row))

#####E###########E###
S.#...#.....#.......#
#.###.#.###.#.#.###.#
#...#.#...#...#...#.#
###.#.###.#######.#.#
#.#.#.....#.#.....#.#
#.#...####..#.#####.#
#.#.#...#...#.#...#.#
#.#.###.#.#.#.###.#.#
#.#.#.#...#.....#...#
#.#.#.#####.###.#.###
#...#.....#...#.#.#.#
#.#####.#.###.#.#.#.#
#.....#.#.#...#.#...#
#####.#.###.###.#####
#.#...#.......#.#...#
#.#.###########.#.#.#
#...#.........#...#.#
#.###.#######.#####.#
#.....#.............#
#####################


In [ ]:
def parse_maze(maze):
    start = None
    exits = []

    for r in range(len(maze)):
        for c in range(len(maze[0])):
            if maze[r, c] == "S":
                start = (r, c)
            elif maze[r, c] == "E":
                exits.append((r, c))

    return start, exits

In [ ]:
start, exits = parse_maze(maze)

print("Start:", start)
print("Exits:", exits)

Start: (1, 0)
Exits: [(0, 5), (0, 17)]


In [ ]:
def neighbors(maze, position):
    r, c = position

    directions = [
        (-1, 0),  # Up
        (1, 0),   # Down
        (0, -1),  # Left
        (0, 1)    # Right
    ]

    result = []

    for dr, dc in directions:
        nr, nc = r + dr, c + dc

        if (0 <= nr < maze.shape[0] and
            0 <= nc < maze.shape[1] and
            maze[nr, nc] != "#"):

            result.append((nr, nc))

    return result

In [ ]:
print("Start:", start)
print("Valid neighbors:", neighbors(maze, start))

Start: (1, 0)
Valid neighbors: [(1, 1)]


In [ ]:
def reconstruct_path(parent, goal):
    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()

    return path

**BFS**

In [ ]:
from collections import deque

def bfs(maze):
    start, exits = parse_maze(maze)
    exit_set = set(exits)

    queue = deque([start])
    visited = {start}
    parent = {start: None}

    expanded_nodes = 0

    while queue:
        current = queue.popleft()

        if current in exit_set:
            path = reconstruct_path(parent, current)

            return {
                "path": path,
                "path_length": len(path) - 1,
                "expanded_nodes": expanded_nodes
            }

        expanded_nodes += 1

        for next_position in neighbors(maze, current):
            if next_position not in visited:
                visited.add(next_position)
                parent[next_position] = current
                queue.append(next_position)

    return None

In [ ]:
bfs_result = bfs(maze)

print("BFS Path:", bfs_result["path"])
print("Path Length:", bfs_result["path_length"])
print("Expanded Nodes:", bfs_result["expanded_nodes"])

BFS Path: [(1, 0), (1, 1), (2, 1), (3, 1), (3, 2), (3, 3), (4, 3), (5, 3), (6, 3), (6, 4), (6, 5), (5, 5), (4, 5), (3, 5), (2, 5), (1, 5), (0, 5)]
Path Length: 16
Expanded Nodes: 34


**DFS**

In [ ]:
def dfs(maze):
    start, exits = parse_maze(maze)
    exit_set = set(exits)

    stack = [start]
    visited = {start}
    parent = {start: None}

    expanded_nodes = 0

    while stack:
        current = stack.pop()

        if current in exit_set:
            path = reconstruct_path(parent, current)

            return {
                "path": path,
                "path_length": len(path) - 1,
                "expanded_nodes": expanded_nodes
            }

        expanded_nodes += 1

        for next_position in neighbors(maze, current):
            if next_position not in visited:
                visited.add(next_position)
                parent[next_position] = current
                stack.append(next_position)

    return None

In [ ]:
dfs_result = dfs(maze)

print("DFS Path:", dfs_result["path"])
print("Path Length:", dfs_result["path_length"])
print("Expanded Nodes:", dfs_result["expanded_nodes"])

DFS Path: [(1, 0), (1, 1), (2, 1), (3, 1), (3, 2), (3, 3), (4, 3), (5, 3), (6, 3), (6, 4), (6, 5), (7, 5), (7, 6), (7, 7), (8, 7), (9, 7), (9, 8), (9, 9), (8, 9), (7, 9), (7, 10), (7, 11), (8, 11), (9, 11), (9, 12), (9, 13), (8, 13), (7, 13), (6, 13), (5, 13), (5, 14), (5, 15), (5, 16), (5, 17), (4, 17), (3, 17), (3, 16), (3, 15), (2, 15), (1, 15), (1, 16), (1, 17), (0, 17)]
Path Length: 42
Expanded Nodes: 143


**GBFS**

In [ ]:
import heapq

def heuristic(position, exits):
    r, c = position

    return min(
        abs(r - er) + abs(c - ec)
        for er, ec in exits)

def gbfs(maze):
    start, exits = parse_maze(maze)
    exit_set = set(exits)

    priority_queue = []
    heapq.heappush(priority_queue,(heuristic(start, exits), start))

    visited = {start}
    parent = {start: None}

    expanded_nodes = 0

    while priority_queue:
        _, current = heapq.heappop(priority_queue)

        if current in exit_set:
            path = reconstruct_path(parent, current)

            return {"path": path,
                    "path_length": len(path) - 1,
                    "expanded_nodes": expanded_nodes}
        expanded_nodes += 1

        for next_position in neighbors(maze, current):
            if next_position not in visited:
                visited.add(next_position)
                parent[next_position] = current

                h = heuristic(next_position, exits)

                heapq.heappush(
                    priority_queue,
                    (h, next_position))
    return None

In [ ]:
gbfs_result = gbfs(maze)

print("GBFS Path:", gbfs_result["path"])
print("Path Length:", gbfs_result["path_length"])
print("Expanded Nodes:", gbfs_result["expanded_nodes"])


GBFS Path: [(1, 0), (1, 1), (2, 1), (3, 1), (3, 2), (3, 3), (4, 3), (5, 3), (6, 3), (6, 4), (6, 5), (5, 5), (4, 5), (3, 5), (2, 5), (1, 5), (0, 5)]
Path Length: 16
Expanded Nodes: 16


In [ ]:
def measure_algorithm(algorithm, maze):

    # Measure execution time
    start_time = time.perf_counter()
    result = algorithm(maze)
    end_time = time.perf_counter()

    execution_time = (end_time - start_time) * 1000

    # Measure peak memory usage separately
    tracemalloc.start()

    memory_result = algorithm(maze)

    current_memory, peak_memory = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    peak_memory_kb = peak_memory / 1024

    return {
        "path_length": result["path_length"],
        "expanded_nodes": result["expanded_nodes"],
        "execution_time_ms": execution_time,
        "peak_memory_kb": peak_memory_kb}

**BENCHMARKING**

In [ ]:
def benchmark_all():
    configurations = [(10, 2),(25, 4),(50, 6),(100, 10)]
    algorithms = {
        "BFS": bfs,
        "DFS": dfs,
        "GBFS": gbfs}
    results = []
    for dim, num_exits in configurations:
        print(f"\nTesting maze size: {2*dim+1}x{2*dim+1}")
        for seed in range(100):
            # Generate maze
            maze = create_maze(dim, num_exits, seed)
            # BFS provides the shortest-path baseline
            bfs_baseline = bfs(maze)
            if bfs_baseline is None:
                print(f"Warning: No path found for seed {seed}")
                continue
            shortest_length = bfs_baseline["path_length"]
            # Test all algorithms
            for name, algorithm in algorithms.items():
                metrics = measure_algorithm(algorithm, maze)
                if metrics is None:
                    continue
                # Check shortest-path rate
                is_shortest = (metrics["path_length"] == shortest_length)
                results.append({
                    "Maze_Size": 2*dim+1,
                    "Exits": num_exits,
                    "Seed": seed,
                    "Algorithm": name,
                    "Path_Length": metrics["path_length"],
                    "Expanded_Nodes": metrics["expanded_nodes"],
                    "Execution_Time_ms": metrics["execution_time_ms"],
                    "Peak_Memory_KB": metrics["peak_memory_kb"],
                    "Is_Shortest": is_shortest})
        print("Completed!")
    return pd.DataFrame(results)

In [ ]:
raw_results = benchmark_all()
print("Total experimental records:", len(raw_results))
raw_results.head()


Testing maze size: 21x21
Completed!

Testing maze size: 51x51
Completed!

Testing maze size: 101x101
Completed!

Testing maze size: 201x201
Completed!
Total experimental records: 1200


,Maze_Size,Exits,Seed,Algorithm,Path_Length,Expanded_Nodes,Execution_Time_ms,Peak_Memory_KB,Is_Shortest
0,21,2,0,BFS,7,7,0.23538,2.312500,True
1,21,2,0,DFS,128,140,0.63194,15.866211,False
2,21,2,0,GBFS,7,7,0.25448,1.898438,True
3,21,2,1,BFS,44,88,0.44184,16.453125,True
4,21,2,1,DFS,76,76,0.42617,12.742188,False


In [ ]:
summary = raw_results.groupby(
    ["Maze_Size", "Algorithm"]
).agg(
    Avg_Path_Length=("Path_Length", "mean"),
    Avg_Expanded_Nodes=("Expanded_Nodes", "mean"),
    Avg_Execution_Time_ms=("Execution_Time_ms", "mean"),
    Avg_Peak_Memory_KB=("Peak_Memory_KB", "mean"),
    Shortest_Path_Rate=("Is_Shortest", "mean")
).reset_index()

summary["Shortest_Path_Rate"] *= 100

summary

,Maze_Size,Algorithm,Avg_Path_Length,Avg_Expanded_Nodes,Avg_Execution_Time_ms,Avg_Peak_Memory_KB,Shortest_Path_Rate
0,21,BFS,44.71,66.93,0.405233,9.558281,100.0
1,21,DFS,81.47,117.70,0.565152,14.360879,22.0
2,21,GBFS,45.68,49.32,0.423172,6.876826,91.0
3,51,BFS,140.28,250.41,2.677884,31.327910,100.0
4,51,DFS,362.92,509.95,3.597655,56.390469,15.0
5,51,GBFS,144.02,177.60,2.658299,22.021172,91.0
6,101,BFS,299.92,653.13,7.438540,80.506553,100.0
7,101,DFS,995.84,1348.93,9.848748,185.886016,11.0
8,101,GBFS,313.26,455.25,7.816974,55.093672,83.0
9,201,BFS,564.80,1277.85,25.646810,189.157012,100.0
